# Módulo 1 — Generadores y Streams de Datos en Python
Curso: Python para Ingeniería de Datos

**Objetivo de este notebook:** entender *qué es un generador*, por qué ayuda con **memoria** y **procesamiento de archivos grandes**, y cómo armar *pipelines* (streams) en Python puro usando `yield`.

> Este notebook está diseñado para Databricks / Colab / Jupyter.


## 1) ¿Qué es un generador?
Un **generador** produce datos **uno por uno** (bajo demanda), en lugar de construir todo en memoria de golpe.

- Función normal: suele devolver todo con `return`.
- Generador: usa **`yield`** y puede **pausarse** y **reanudar** su ejecución.

### Analogía cotidiana
**Lista = bandeja completa**: te traen 1000 panes juntos.  
**Generador = pan por pan**: te dan 1 pan cada vez que lo pides.

En Ingeniería de Datos, esto es clave para:
- Archivos grandes (GBs)
- Streams / procesamiento incremental
- Reducir memoria y mejorar estabilidad


## 2) Conceptos clave
- **`yield`**: entrega un valor y *pausa* la función, guardando el estado interno.
- **`next(gen)`**: pide el siguiente valor del generador (reanuda desde donde se quedó).
- **Iteración (`for`)**: internamente usa `next()` hasta que el generador se termina (`StopIteration`).
- **Generator expression**: parecido a list comprehension, pero con paréntesis `(...)` produce un generador.


In [0]:
# ✅ Ejemplo 0: función normal vs generador

def lista_1_a_5():
    return [1, 2, 3, 4, 5]
print("Función normal:", lista_1_a_5())



In [0]:
def gen_1_a_5():
    for i in range(1, 6):
        yield i  # entrega i y pausa

g = gen_1_a_5()
print("Generador (objeto):", g)
print("Consumir generador con for:")
for x in g:
    print(x)

## 3) ¿Qué hace `yield` exactamente?
Veamos cómo la función “se pausa” y luego continúa.

- La primera llamada a `next(gen)` corre hasta el primer `yield`.
- La siguiente llamada continúa desde justo después de ese `yield`.


`next(gen)` llama al generador y obtiene el siguiente valor producido por `yield`. Si el generador ya terminó, lanza una excepción `StopIteration`.

In [0]:
def demo_yield():
    print("Inicio de la función")
    yield 1
    print("Luego del primer yield")
    yield 2
    print("Luego del segundo yield")
    yield 3
    print("Fin de la función")

gen = demo_yield()

print("next 1:", next(gen))
print("next 2:", next(gen))
print("next 3:", next(gen))

# Si pides uno más, se termina y lanzará StopIteration


In [0]:
print("next 4:", next(gen))

## 4) Lista vs Generador: memoria
La idea no es memorizar números, sino **evitar cargar todo** cuando el volumen es grande.

### Ojo
- `sys.getsizeof` no mide toda la memoria "real" de estructuras complejas, pero sirve para comparar rápido.
- Diferencia conceptual: la lista contiene todo; el generador no.


In [0]:
import sys

lista = [i for i in range(1_000_000)]      # construye 1M elementos
gen = (i for i in range(1_000_000))        # NO construye 1M, solo "sabe cómo producirlos"

print("Memoria lista (bytes):", sys.getsizeof(lista))
print("Memoria generador (bytes):", sys.getsizeof(gen))


**¿Qué significa este resultado?**

- **Memoria lista (bytes): 8448728**  
  La lista ocupa ~8.4 MB porque almacena *todos* los 1,000,000 números en memoria.

- **Memoria generador (bytes): 200**  
  El generador ocupa solo 200 bytes porque *no almacena* los números; solo guarda el estado interno para producir el siguiente valor bajo demanda.

**Conclusión:**  
El generador es mucho más eficiente en memoria, ideal para manejar grandes volúmenes de datos.

## 5) Streams de datos (processing en flujo)
Un **stream** es procesar datos a medida que llegan, sin esperar a tenerlo todo.

En DE, esto se traduce en:
- Leer archivos línea por línea
- Procesar registros por chunks
- Encadenar transformaciones sin listas intermedias


### 5.1) Leer un archivo grande como stream (línea por línea)
Evita `read()` o `readlines()` en archivos grandes.


In [0]:
# ✅ Creamos un archivo de ejemplo (auto-contenido)
import random

path = "ventas_stream.csv"
with open(path, "w") as f:
    for i in range(20):
        # Simulamos algunas líneas malas
        if random.random() < 0.1:
            f.write("BAD_LINE\n")
        else:
            f.write(f"{i},{random.uniform(10,1000):.2f}\n")

print("Archivo generado en:", path)


In [0]:
def leer_archivo_stream(path):
    """Lee un archivo línea por línea (stream) sin cargar todo en memoria."""
    with open(path, "r") as f:
        for linea in f:
            yield linea.strip()  # yield: entrega la línea y pausa

for linea in leer_archivo_stream(path):
    print("LINEA:", linea)


## 6) Pipeline (stream) con generadores encadenados
Construimos un pipeline estilo Spark, pero en Python puro:

1) `leer` (stream)  
2) `validar` (filtrar líneas malas)  
3) `transformar` (parsear campos)  
4) `enriquecer` (agregar algo)  
5) `consumir` (for, guardar, etc.)

**Clave:** no usar listas intermedias; cada paso hace `yield`.


In [0]:
def filtrar_lineas_validas(stream_lineas):
    """Filtra líneas inválidas (por ejemplo 'BAD_LINE')."""
    for linea in stream_lineas:
        if linea == "BAD_LINE" or "," not in linea:
            continue
        yield linea

def parsear_venta(stream_lineas):
    """Transforma 'id,monto' a dict."""
    for linea in stream_lineas:
        id_str, monto_str = linea.split(",")
        yield {"id": int(id_str), "monto": float(monto_str)}

def enriquecer(stream_registros):
    """Agrega una etiqueta de rango de monto."""
    for r in stream_registros:
        r["rango"] = "ALTO" if r["monto"] >= 500 else "BAJO"
        yield r

stream = leer_archivo_stream(path)
stream = filtrar_lineas_validas(stream)
stream = parsear_venta(stream)
stream = enriquecer(stream)

for registro in stream:
    print(registro)


## 7) Comparación conceptual: lista vs stream (archivo pequeño)
En archivos enormes, la ventaja principal del streaming es **memoria + estabilidad** (y empezar a procesar antes).
Aquí comparamos en pequeño para entender la forma.


In [0]:
import time

# A) Cargar todo (NO recomendado en grande)
inicio = time.time()
with open(path, "r") as f:
    lineas = [l.strip() for l in f]
validas = [l for l in lineas if l != "BAD_LINE" and "," in l]
registros = []
for l in validas:
    i, m = l.split(",")
    registros.append({"id": int(i), "monto": float(m)})
fin = time.time()
print(f"⏱ (A) Lista completa: {fin - inicio:.6f} s | registros={len(registros)}")

# B) Stream
inicio = time.time()
stream = enriquecer(parsear_venta(filtrar_lineas_validas(leer_archivo_stream(path))))
registros2 = list(stream)  # solo para contar/mostrar
fin = time.time()
print(f"⏱ (B) Streaming:      {fin - inicio:.6f} s | registros={len(registros2)}")


## 8) Ejemplos extra (súper sencillos)


### 8.1) Generator expression (paréntesis)
- Lista: `[x*x for x in range(10)]`
- Generador: `(x*x for x in range(10))`


In [0]:
lista = [x*x for x in range(10)]     # crea todo de golpe
gen = (x*x for x in range(10))       # produce bajo demanda

print("Lista:", lista)
print("Generador:", gen)

print("Consumir generador:")
for v in gen:
    print(v)


### 8.2) Procesar “muchos registros” sin guardarlos (solo contar)
A veces solo quieres contar/validar/loggear sin almacenar todo.


In [0]:
def numeros_grandes(n):
    for i in range(n):
        yield i

inicio = time.time()
conteo = 0
for _ in numeros_grandes(1_000_000):
    conteo += 1
fin = time.time()

print("Conteo:", conteo)
print(f"⏱ Tiempo recorriendo 1M en stream: {fin - inicio:.2f} s")


---
# 9) Ejercicios (para clase)


## Ejercicio 1 — Lista vs Generador (memoria)
1) Crea una lista con 1,000,000 números.  
2) Crea un generador con 1,000,000 números.  
3) Imprime `sys.getsizeof` de ambos.  
4) Explica por qué el generador consume menos.


In [0]:
# ✅ Tu solución aquí
import sys

# TODO: lista
# TODO: generador

# print(sys.getsizeof(...))


## Ejercicio 2 — Stream de archivo
Crea un archivo con 100 líneas 'id,monto' y algunas líneas malas 'BAD_LINE'.
Luego:
- Lee como stream con `yield`
- Filtra malas
- Parsear a dict
- Cuenta registros válidos


In [0]:
# ✅ Tu solución aquí
# 1) Generar archivo
# 2) leer_stream()
# 3) filtrar()
# 4) parsear()
# 5) contar


## Ejercicio 3 — Pipeline encadenado (sin listas intermedias)
Construye un pipeline para números 1..100:
- Generar números (yield)
- Filtrar múltiplos de 3
- Elevar al cuadrado
- Muestra los primeros 10 resultados

**Prohibido:** listas intermedias en los pasos (solo `yield`).


In [0]:
# ✅ Tu solución aquí
# def generar():
# def filtrar_multiplos_de_3(stream):
# def cuadrados(stream):
# ...
# Consumir y mostrar primeros 10


---
# Checklist mental
- Generadores (`yield`) = producir datos bajo demanda.
- Streams = procesar incrementalmente (línea por línea / chunk por chunk).
- Evitas reventar memoria y mejoras estabilidad.
- Pipelines encadenados = estilo “Spark” pero en Python puro.
